In [ ]:
import sqlite3
import datetime
import gradio as gr
import smtplib
from scraper import fetch_website_contents, fetch_website_links

In [ ]:
conn = sqlite3.connect("../data/finance.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS salary (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amount REAL,
    month TEXT,
    year INTEGER,
    created_at TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amount REAL,
    category TEXT,
    description TEXT,
    date TEXT,
    created_at TEXT
)
""")

conn.commit()
print("Database Ready")

In [ ]:
def extract_amount(text):
    for word in text.split():
        if word.isdigit():
            return int(word)
    return None


def add_expense(amount, category):
    now = datetime.datetime.now()
    date = str(now.date())

    cursor.execute(
        "INSERT INTO expenses (amount, category, date, created_at) VALUES (?, ?, ?, ?)",
        (amount, category, date, str(now))
    )
    conn.commit()

def add_salary(amount):
    now = datetime.datetime.now()

    cursor.execute(
        "INSERT INTO salary (amount, month, year, created_at) VALUES (?, ?, ?, ?)",
        (amount, now.strftime("%B"), now.year, str(now))
    )
    conn.commit()

In [ ]:
def get_expenses():
    cursor.execute("SELECT * FROM expenses")
    return cursor.fetchall()

In [ ]:
def get_balance():
    
    cursor.execute("SELECT SUM(amount) FROM expenses")
    expenses = cursor.fetchone()[0] or 0

    
    cursor.execute("SELECT amount FROM salary ORDER BY id DESC LIMIT 1")
    result = cursor.fetchone()
    salary = result[0] if result else 0

    balance = salary - expenses
    return salary, expenses, balance


In [ ]:
def today_summary():
    today = str(datetime.date.today())

    cursor.execute("SELECT * FROM expenses WHERE date=?", (today,))
    data = cursor.fetchall()

    total = sum([row[1] for row in data])
    return f"Today's spending: Rs.{total}"

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

def generate_pdf(summary):
    file_path = "finance_report.pdf"

    doc = SimpleDocTemplate(file_path)
    styles = getSampleStyleSheet()

    content = []
    for line in summary.split("\n"):
        content.append(Paragraph(line, styles["Normal"]))

    doc.build(content)

    return file_path

In [ ]:
def send_email(summary, receiver, sender, password):
    if not receiver.strip() or not sender.strip() or not password.strip():
        return False, "Please fill in all email fields in the sidebar."
    try:
        message = f"Subject: Finance Report\n\n{summary}"
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(sender.strip(), password.strip())
        server.sendmail(sender.strip(), receiver.strip(), message)
        server.quit()
        return True, "OK"
    except Exception as e:
        return False, str(e)


In [ ]:
def chatbot(message,history,customer_email, sender_email, sender_password):
    text = message.lower()

    if "spent" in text:
        amount = extract_amount(text)
        if amount is not None:
            category = text.split("on")[-1].strip().split()[0] if "on" in text else "general"
            add_expense(amount, category)
            return f"Rs.{amount} added to {category}"
        return "Try: 'I spent 500 on food'"

    if "salary" in text:
        amount = extract_amount(text)
        if amount is not None:
            add_salary(amount)
            return f"Salary recorded: Rs.{amount}"
        return "Try: 'My salary is 50000'"

    if "balance" in text or "how much" in text:
        salary, expenses, balance = get_balance()
        return (
            f"\U0001f4b0 Balance Summary\n"
            f"Salary:   Rs. {int(salary):,}\n"
            f"Expenses: Rs. {int(expenses):,}\n"
            f"Balance:  Rs. {int(balance):,}"
        )

    if "today" in text:
        return today_summary()

    if "http" in text or "fetch" in text or "scrape" in text:
        url = next((w.strip(".,") for w in message.split() if w.startswith("http")), None)
        if not url:
            return "Please include a URL."
        if "link" in text:
            links = fetch_website_links(url)
            return f"Found {len(links)} links:\n" + "\n".join(links[:10])
        return f"Website content:\n\n{fetch_website_contents(url)}"

    if "email" in text:
        salary, expenses, balance = get_balance()
        summary = (
            f"Finance Report\n"
            f"Salary:   Rs.{int(salary):,}\n"
            f"Expenses: Rs.{int(expenses):,}\n"
            f"Balance:  Rs.{int(balance):,}"
        )
        success, msg = send_email(summary, customer_email, sender_email, sender_password)
        return f"\u2705 Report sent to {customer_email.strip()}!" if success else f"\u274c {msg}"

    if "report" in text or "pdf" in text:
        salary, expenses, balance = get_balance()
        summary = (
            f"Finance Report\n"
            f"Salary: Rs.{salary}\n"
            f"Expenses: Rs.{expenses}\n"
            f"Balance: Rs.{balance}"
        )
        return generate_pdf(summary)

    return "Commands: 'I spent 500 on food', 'My salary is 50000', 'How much left?', 'Send email report'"


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

with gr.Blocks(title="Finance Tracker") as app:

    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown("## Finance Dashboard")
            gr.Markdown("Track your income and expenses")

            customer_email = gr.Textbox(
                label="Recipient Email",
                placeholder="Email to receive the report",
                value=""
            )
            sender_email = gr.Textbox(
                label="Your Gmail",
                placeholder="",
                value=os.getenv("SENDER_EMAIL", "")
            )
            sender_password = gr.Textbox(
                label="Gmail App Password",
                placeholder="xxxx xxxx xxxx xxxx",
                type="password",
                value=os.getenv("GMAIL_APP_PASSWORD", "")
            )

        with gr.Column(scale=2):         
            gr.ChatInterface(
                fn=chatbot,
                additional_inputs=[customer_email, sender_email, sender_password],  
                examples=[
                    ["My salary is 50000"],
                    ["I spent 200 on food"],
                    ["How much do I have left?"],
                    ["Send email report"],
                ],
            )

app.launch()